In [ ]:
import pandas as pd
import geopandas as gpd
import plotly.express as px
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
#from palettable.colorbrewer.sequential import Oranges_9, PuRd_9, YlGnBu_9, PuBuGn_9, YlOrRd_9, RdPu_7
#from palettable.colorbrewer.qualitative import Paired_12 
#from palettable.colorbrewer.diverging import RdYlGn_7, Spectral_9, PiYG_6

from lonboard import Map, PolygonLayer, ScatterplotLayer
from lonboard.colormap import apply_continuous_cmap, apply_categorical_cmap
import pydeck

import h3pandas
import h3

from functools import partial
import colorcet as cc

import os, shutil, glob
import time
import warnings
warnings.simplefilter("ignore")
import numpy as np


# 1. Read the classification (primary;category)
classification = pd.read_csv("./data/classification_paper.txt", sep=";", names=["primary", "category"])

import h3
from shapely.geometry import Polygon


In [ ]:
# 2. Read the POI file
category_list = pd.read_csv("./data/category_list.csv")


In [ ]:

# 3. Join the two datasets on "primary"
merged = category_list.merge(classification, on="primary", how="left")

In [ ]:
merged.head(35)

In [ ]:
fin_pois = gpd.read_parquet('output/pois_finland.parquet')

In [ ]:
# Extract `primary` and `alternate` columns
def extract_categories(cat):
    if isinstance(cat, dict):
        primary = cat.get('primary', None)
        alternate = cat.get('alternate', [])
        if not isinstance(alternate, list):
            alternate = []
    else:
        primary = None
        alternate = []
    # pad alternates to length 2
    alternate = (alternate + [None, None])[:2]
    return pd.Series([primary] + alternate)

fin_pois[['primary', 'alternate_1', 'alternate_2']] = fin_pois['categories'].apply(extract_categories)



In [ ]:
import numpy as np

def extract_categories(cat):
    if isinstance(cat, dict):
        primary = cat.get('primary', None)
        alternate = cat.get('alternate', None)

        alt1, alt2 = None, None
        if isinstance(alternate, np.ndarray):
            if len(alternate) > 0:
                alt1 = alternate[0]
            if len(alternate) > 1:
                alt2 = alternate[1]
        elif isinstance(alternate, list):
            if len(alternate) > 0:
                alt1 = alternate[0]
            if len(alternate) > 1:
                alt2 = alternate[1]

        return pd.Series([primary, alt1, alt2])
    else:
        return pd.Series([None, None, None])

fin_pois[['primary', 'alternate_1', 'alternate_2']] = fin_pois['categories'].apply(extract_categories)


In [ ]:
geo_turku = gpd.read_file('./data/Turku_region_boundary.geojson')

In [ ]:
geo_hsk = gpd.read_file('./data/h3_polygons_Helsinki_whole.gpkg')

In [ ]:
geo_tampere = gpd.read_file('./data/Tampere_region_boundary.geojson')

In [ ]:
geo_oulu = gpd.read_file('./data/oulu_region_boundary.geojson')

In [ ]:
# Dissolve into one polygon
geo_hsk = geo_hsk.dissolve()

In [ ]:
# Dissolve into one polygon
geo_turku = geo_turku.dissolve()

In [ ]:
# Dissolve into one polygon
geo_tampere = geo_tampere.dissolve()

In [ ]:
# Dissolve into one polygon
geo_oulu = geo_oulu.dissolve()

In [ ]:
fin_pois = fin_pois.set_crs("EPSG:4326", inplace=False)

In [ ]:
fin_pois = fin_pois.set_crs("EPSG:4326", inplace=False)
geo_hsk = geo_hsk.set_crs("EPSG:4326", inplace=False)

In [ ]:
geo_turku = geo_turku.set_crs("EPSG:4326", inplace=False)

In [ ]:
geo_tampere = geo_tampere.set_crs("EPSG:4326", inplace=False)

In [ ]:
geo_oulu = geo_oulu.set_crs("EPSG:4326", inplace=False)

In [ ]:
# Spatial join or mask
fin_pois_hsk = gpd.sjoin(fin_pois, geo_hsk, how="inner", predicate="within")

In [ ]:
# Spatial join or mask
fin_pois_turku = gpd.sjoin(fin_pois, geo_turku, how="inner", predicate="within")

In [ ]:
fin_pois_tampere = gpd.sjoin(fin_pois, geo_tampere, how="inner", predicate="within")

In [ ]:
fin_pois_oulu= gpd.sjoin(fin_pois, geo_oulu, how="inner", predicate="within")

In [ ]:
fin_pois_hsk.shape

In [ ]:
fin_pois_turku.shape

In [ ]:
fin_pois_tampere.shape

In [ ]:
fin_pois_oulu.shape

In [ ]:
fin_pois_hsk = fin_pois_hsk[fin_pois_hsk["confidence"] >= 0.50] #29.5K

In [ ]:
fin_pois_hsk

In [ ]:
fin_pois_turku = fin_pois_turku[fin_pois_turku["confidence"] >= 0.50] #8.25K

In [ ]:
fin_pois_turku

In [ ]:
fin_pois_tampere = fin_pois_tampere[fin_pois_tampere["confidence"] >= 0.50] #8.25K

In [ ]:
fin_pois_tampere

In [ ]:
fin_pois_oulu = fin_pois_oulu[fin_pois_oulu["confidence"] >= 0.50] #6.1K

In [ ]:
fin_pois_oulu

In [ ]:
n_unique_categories = fin_pois_hsk['primary'].nunique()
print(f"Unique primary categories: {n_unique_categories}")

In [ ]:
# 3. Join the two datasets on "primary"
fin_pois_hsk = fin_pois_hsk.merge(classification, on="primary", how="left")

In [ ]:
fin_pois_turku = fin_pois_turku.merge(classification, on="primary", how="left")

In [ ]:
fin_pois_tampere = fin_pois_tampere.merge(classification, on="primary", how="left")

In [ ]:
fin_pois_oulu = fin_pois_oulu.merge(classification, on="primary", how="left")

In [ ]:
import geopandas as gpd
import folium
from branca.colormap import linear

# Ensure CRS is WGS84
if fin_pois_hsk.crs != "EPSG:4326":
    fin_pois_hsk = fin_pois_hsk.to_crs(epsg=4326)

# Unique categories and color map
categories = sorted(fin_pois_hsk['category'].dropna().unique())
colors = linear.Set1_09.scale(0, len(categories)).to_step(n=len(categories))
color_map = {cat: colors.rgb_hex_str(i) for i, cat in enumerate(categories)}

# Base map
center = [
    fin_pois_hsk.geometry.y.mean(),
    fin_pois_hsk.geometry.x.mean()
]
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

# Add base layers
folium.TileLayer('OpenStreetMap').add_to(m)
folium.TileLayer('CartoDB dark_matter').add_to(m)

# Add points
for idx, row in fin_pois_hsk.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color=color_map.get(row['category'], 'gray'),
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(str(row['category']), parse_html=True)
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Add legend
legend_html = """
<div style='position: fixed; bottom: 50px; left: 50px; width: 200px; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:14px;'>
&nbsp;<b>Categories</b><br>
"""
for cat, col in color_map.items():
    legend_html += f"&nbsp;<i style='background:{col}'>&nbsp;&nbsp;&nbsp;&nbsp;</i> {cat}<br>"
legend_html += "</div>"

m.get_root().html.add_child(folium.Element(legend_html))

#m.save("./output/pois_by_category_map_no_cluster.html")


In [ ]:
import geopandas as gpd
import folium
from branca.colormap import linear

# Ensure CRS is WGS84
if fin_pois_turku.crs != "EPSG:4326":
    fin_pois_turku = fin_pois_turku.set_crs(epsg=4326)

# Unique categories and color map
categories = sorted(fin_pois_turku['category'].dropna().unique())
colors = linear.Set1_09.scale(0, len(categories)).to_step(n=len(categories))
color_map = {cat: colors.rgb_hex_str(i) for i, cat in enumerate(categories)}

# Base map
center = [
    fin_pois_turku.geometry.y.mean(),
    fin_pois_turku.geometry.x.mean()
]

m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

# Add base layers
folium.TileLayer('OpenStreetMap').add_to(m)
folium.TileLayer('CartoDB dark_matter').add_to(m)

# Add points
for idx, row in fin_pois_turku.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color=color_map.get(row['category'], 'gray'),
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(str(row['category']), parse_html=True)
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Add legend
legend_html = """
<div style='position: fixed; bottom: 50px; left: 50px; width: 200px; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:14px;'>
&nbsp;<b>Categories</b><br>
"""

for cat, col in color_map.items():
    legend_html += f"&nbsp;<i style='background:{col}'>&nbsp;&nbsp;&nbsp;&nbsp;</i> {cat}<br>"

legend_html += "</div>"

m.get_root().html.add_child(folium.Element(legend_html))

# Save map
#m.save("./output/pois_by_category_map_turku.html")

In [ ]:
# Ensure CRS is WGS84
if fin_pois_tampere.crs != "EPSG:4326":
    fin_pois_tampere = fin_pois_tampere.set_crs(epsg=4326)

# Unique categories and color map
categories = sorted(fin_pois_tampere['category'].dropna().unique())
colors = linear.Set1_09.scale(0, len(categories)).to_step(n=len(categories))
color_map = {cat: colors.rgb_hex_str(i) for i, cat in enumerate(categories)}

# Base map
center = [
    fin_pois_tampere.geometry.y.mean(),
    fin_pois_tampere.geometry.x.mean()
]

m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

# Add base layers
folium.TileLayer('OpenStreetMap').add_to(m)
folium.TileLayer('CartoDB dark_matter').add_to(m)

# Add points
for idx, row in fin_pois_tampere.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color=color_map.get(row['category'], 'gray'),
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(str(row['category']), parse_html=True)
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Add legend
legend_html = """
<div style='position: fixed; bottom: 50px; left: 50px; width: 200px; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:14px;'>
&nbsp;<b>Categories</b><br>
"""

for cat, col in color_map.items():
    legend_html += f"&nbsp;<i style='background:{col}'>&nbsp;&nbsp;&nbsp;&nbsp;</i> {cat}<br>"

legend_html += "</div>"

m.get_root().html.add_child(folium.Element(legend_html))

# Save map
m.save("./output/pois_by_category_map_tampere.html")

In [ ]:
import geopandas as gpd
import folium
from branca.colormap import linear

# Ensure CRS is WGS84
if fin_pois_oulu.crs != "EPSG:4326":
    fin_pois_oulu = fin_pois_oulu.set_crs(epsg=4326)

# Unique categories and color map
categories = sorted(fin_pois_oulu['category'].dropna().unique())
colors = linear.Set1_09.scale(0, len(categories)).to_step(n=len(categories))
color_map = {cat: colors.rgb_hex_str(i) for i, cat in enumerate(categories)}

# Base map
center = [
    fin_pois_oulu.geometry.y.mean(),
    fin_pois_oulu.geometry.x.mean()
]

m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

# Add base layers
folium.TileLayer('OpenStreetMap').add_to(m)
folium.TileLayer('CartoDB dark_matter').add_to(m)

# Add points
for idx, row in fin_pois_oulu.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color=color_map.get(row['category'], 'gray'),
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(str(row['category']), parse_html=True)
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Add legend
legend_html = """
<div style='position: fixed; bottom: 50px; left: 50px; width: 200px; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:14px;'>
&nbsp;<b>Categories</b><br>
"""

for cat, col in color_map.items():
    legend_html += f"&nbsp;<i style='background:{col}'>&nbsp;&nbsp;&nbsp;&nbsp;</i> {cat}<br>"

legend_html += "</div>"

m.get_root().html.add_child(folium.Element(legend_html))

# Save map
m.save("./output/pois_by_category_map_oulu.html")

In [ ]:
#pip install "h3<4"

In [ ]:
import h3
print(h3.__version__)

In [ ]:
# Create H3 hexagons (resolution 9)
def polyfill_geometry(geom, res):
    return list(h3.polyfill(geom.__geo_interface__, res, geo_json_conformant=True))

hex_ids = set()
for geom in geo_hsk.geometry:
    hex_ids.update(polyfill_geometry(geom, 9))

# Build GeoDataFrame of hexagons
h3_geoms = [Polygon(h3.h3_to_geo_boundary(h, geo_json=True)) for h in hex_ids]
h3_hsk = gpd.GeoDataFrame({'h3_id': list(hex_ids), 'geometry': h3_geoms}, crs='EPSG:4326')

In [ ]:
geo_turku

In [ ]:
from shapely.geometry import Polygon, MultiPolygon

def polyfill_geometry(geom, res):
    if geom.geom_type == "Polygon":
        return list(h3.polyfill(geom.__geo_interface__, res, geo_json_conformant=True))
    
    elif geom.geom_type == "MultiPolygon":
        hexes = []
        for poly in geom.geoms:
            hexes.extend(h3.polyfill(poly.__geo_interface__, res, geo_json_conformant=True))
        return hexes

hex_ids = set()

for geom in geo_turku.geometry:
    hex_ids.update(polyfill_geometry(geom, 9))

# Build GeoDataFrame of hexagons
h3_geoms = [Polygon(h3.h3_to_geo_boundary(h, geo_json=True)) for h in hex_ids]

h3_turku = gpd.GeoDataFrame(
    {"h3_id": list(hex_ids), "geometry": h3_geoms},
    crs="EPSG:4326"
)

In [ ]:
def polyfill_geometry(geom, res):
    if geom.geom_type == "Polygon":
        return list(h3.polyfill(geom.__geo_interface__, res, geo_json_conformant=True))
    
    elif geom.geom_type == "MultiPolygon":
        hexes = []
        for poly in geom.geoms:
            hexes.extend(h3.polyfill(poly.__geo_interface__, res, geo_json_conformant=True))
        return hexes

hex_ids = set()

for geom in geo_tampere.geometry:
    hex_ids.update(polyfill_geometry(geom, 9))

# Build GeoDataFrame of hexagons
h3_geoms = [Polygon(h3.h3_to_geo_boundary(h, geo_json=True)) for h in hex_ids]

h3_tampere = gpd.GeoDataFrame(
    {"h3_id": list(hex_ids), "geometry": h3_geoms},
    crs="EPSG:4326"
)

In [ ]:
def polyfill_geometry(geom, res):
    if geom.geom_type == "Polygon":
        return list(h3.polyfill(geom.__geo_interface__, res, geo_json_conformant=True))
    
    elif geom.geom_type == "MultiPolygon":
        hexes = []
        for poly in geom.geoms:
            hexes.extend(h3.polyfill(poly.__geo_interface__, res, geo_json_conformant=True))
        return hexes

hex_ids = set()

for geom in geo_oulu.geometry:
    hex_ids.update(polyfill_geometry(geom, 9))

# Build GeoDataFrame of hexagons
h3_geoms = [Polygon(h3.h3_to_geo_boundary(h, geo_json=True)) for h in hex_ids]

h3_oulu = gpd.GeoDataFrame(
    {"h3_id": list(hex_ids), "geometry": h3_geoms},
    crs="EPSG:4326"
)

In [ ]:
# Check for conflicting column before join
if 'index_right' in fin_pois_hsk.columns:
    fin_pois_hsk = fin_pois_hsk.drop(columns='index_right')

In [ ]:
# Check for conflicting column before join
if 'index_right' in fin_pois_turku.columns:
    fin_pois_turku = fin_pois_turku.drop(columns='index_right')

In [ ]:
# Check for conflicting column before join
if 'index_right' in fin_pois_tampere.columns:
    fin_pois_tampere = fin_pois_tampere.drop(columns='index_right')

In [ ]:
# Check for conflicting column before join
if 'index_right' in fin_pois_oulu.columns:
    fin_pois_oulu = fin_pois_oulu.drop(columns='index_right')

In [ ]:
# Spatial join with POIs (keeping only those in fin_pois_hsk)
pois_with_h3 = gpd.sjoin(fin_pois_hsk, h3_hsk, predicate="intersects", how="left")


In [ ]:
# Spatial join with POIs (keeping only those in fin_pois_hsk)
pois_with_h3_turku = gpd.sjoin(fin_pois_turku, h3_turku, predicate="intersects", how="left")


In [ ]:
# Spatial join with POIs (keeping only those in fin_pois_hsk)
pois_with_h3_tampere = gpd.sjoin(fin_pois_tampere, h3_tampere, predicate="intersects", how="left")


In [ ]:
# Spatial join with POIs (keeping only those in fin_pois_hsk)
pois_with_h3_oulu = gpd.sjoin(fin_pois_oulu, h3_oulu, predicate="intersects", how="left")


In [ ]:
# Group by h3_id and category, then count how many POIs per category per hexagon
category_hsk_freq = pois_with_h3.groupby(["h3_id", "category"]).size().reset_index(name="count")

In [ ]:
# Group by h3_id and category, then count how many POIs per category per hexagon
category_turku_freq = pois_with_h3_turku.groupby(["h3_id", "category"]).size().reset_index(name="count")

In [ ]:
# Group by h3_id and category, then count how many POIs per category per hexagon
category_tampere_freq = pois_with_h3_tampere.groupby(["h3_id", "category"]).size().reset_index(name="count")

In [ ]:
# Group by h3_id and category, then count how many POIs per category per hexagon
category_oulu_freq = pois_with_h3_oulu.groupby(["h3_id", "category"]).size().reset_index(name="count")

In [ ]:
category_hsk_freq.category.unique()

In [ ]:
category_turku_freq.category.unique()

In [ ]:
category_tampere_freq.category.unique()

In [ ]:
category_oulu_freq.category.unique()

In [ ]:
category_hsk_freq.to_parquet("data/pois_per_hex_new_class.parquet")

In [ ]:
category_turku_freq.to_parquet("data/pois_per_hex_new_class_turku.parquet")

In [ ]:
category_tampere_freq.to_parquet("data/pois_per_hex_new_class_tampere.parquet")

In [ ]:
category_oulu_freq.to_parquet("data/pois_per_hex_new_class_oulu.parquet")

In [ ]:
## Work distribution rttk

In [ ]:
census = gpd.read_file("data/grid_data/rttk250m_tilv2019.shp")

In [ ]:
census.columns

In [ ]:
area = gpd.read_file('./data/h3_polygons_Helsinki_whole.gpkg')

In [ ]:
area_turku = gpd.read_file('./data/Turku_region_boundary.geojson')

In [ ]:
area_tampere = gpd.read_file('./data/Tampere_region_boundary.geojson')

In [ ]:
area_oulu = gpd.read_file('./data/oulu_region_boundary.geojson')

In [ ]:
# Reproject area to match census CRS
area_proj = area.to_crs(census.crs)

# Dissolve into one polygon
area_union = area_proj.dissolve()

# Spatial join (inner)
census_clip = gpd.sjoin(census, area_union, how="inner", predicate="intersects")

# Clean index_right if it appears
census_clip = census_clip.drop(columns=["index_right"], errors="ignore")

census_clip.plot()

In [ ]:
# Reproject area to match census CRS
area_turku_proj = area_turku.to_crs(census.crs)

# Dissolve into one polygon
area_turku_union = area_turku_proj.dissolve()

# Spatial join (inner)
census_clip_turku = gpd.sjoin(census, area_turku_union, how="inner", predicate="intersects")

# Clean index_right if it appears
census_clip_turku = census_clip_turku.drop(columns=["index_right"], errors="ignore")

# Plot
census_clip_turku.plot()

In [ ]:
# Reproject area to match census CRS
area_tampere_proj = area_tampere.to_crs(census.crs)

# Dissolve into one polygon
area_tampere_union = area_tampere_proj.dissolve()

# Spatial join (inner)
census_clip_tampere = gpd.sjoin(census, area_tampere_union, how="inner", predicate="intersects")

# Clean index_right if it appears
census_clip_tampere = census_clip_tampere.drop(columns=["index_right"], errors="ignore")

# Plot
census_clip_tampere.plot()

In [ ]:
# Reproject area to match census CRS
area_oulu_proj = area_oulu.to_crs(census.crs)

# Dissolve into one polygon
area_oulu_union = area_oulu_proj.dissolve()

# Spatial join (inner)
census_clip_oulu = gpd.sjoin(census, area_oulu_union, how="inner", predicate="intersects")

# Clean index_right if it appears
census_clip_oulu = census_clip_oulu.drop(columns=["index_right"], errors="ignore")

# Plot
census_clip_oulu.plot()

In [ ]:
census = census_clip.copy()

In [ ]:
census_turku = census_clip_turku.copy()

In [ ]:
census_tampere = census_clip_tampere.copy()

In [ ]:
census_oulu = census_clip_oulu.copy()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import mapclassify

# Make sure the column is numeric
census['tp_tyopy'] = pd.to_numeric(census['tp_tyopy'], errors='coerce')

# Build a natural breaks (Jenks) classifier
nb_class = mapclassify.NaturalBreaks(census['tp_tyopy'], k=7)  # k = number of classes

# Plot the map using natural breaks
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
census.plot(column='tp_tyopy', 
            cmap='viridis', 
            scheme='NaturalBreaks',   # required to use the class breaks
            classification_kwds={'k': 7}, # number of classes
            legend=True, 
            edgecolor='black',
            linewidth=0.3, 
            ax=ax)

ax.set_title("Census map by total jobs (Natural Breaks)", fontsize=15)
ax.axis('off')
plt.show()


In [ ]:

import mapclassify

# Ensure column is numeric
census_clip_turku['tp_tyopy'] = pd.to_numeric(census_clip_turku['tp_tyopy'], errors='coerce')

# Create Natural Breaks classifier
nb_class_turku = mapclassify.NaturalBreaks(census_clip_turku['tp_tyopy'], k=7)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

census_clip_turku.plot(
    column='tp_tyopy',
    cmap='viridis',
    scheme='NaturalBreaks',
    classification_kwds={'k': 7},
    legend=True,
    edgecolor='black',
    linewidth=0.3,
    ax=ax
)

ax.set_title("Turku census map by total jobs (Natural Breaks)", fontsize=15)
ax.axis('off')

plt.show()


In [ ]:
import mapclassify

# Ensure column is numeric
census_clip_tampere['tp_tyopy'] = pd.to_numeric(census_clip_tampere['tp_tyopy'], errors='coerce')

# Create Natural Breaks classifier
nb_class_tampere = mapclassify.NaturalBreaks(census_clip_tampere['tp_tyopy'], k=7)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

census_clip_tampere.plot(
    column='tp_tyopy',
    cmap='viridis',
    scheme='NaturalBreaks',
    classification_kwds={'k': 7},
    legend=True,
    edgecolor='black',
    linewidth=0.3,
    ax=ax
)

ax.set_title("Tampere census map by total jobs (Natural Breaks)", fontsize=15)
ax.axis('off')

plt.show()

In [ ]:
import mapclassify

# Ensure column is numeric
census_clip_oulu['tp_tyopy'] = pd.to_numeric(census_clip_oulu['tp_tyopy'], errors='coerce')

# Create Natural Breaks classifier
nb_class_oulu = mapclassify.NaturalBreaks(census_clip_oulu['tp_tyopy'], k=7)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

census_clip_oulu.plot(
    column='tp_tyopy',
    cmap='viridis',
    scheme='NaturalBreaks',
    classification_kwds={'k': 7},
    legend=True,
    edgecolor='black',
    linewidth=0.3,
    ax=ax
)

ax.set_title("Oulu census map by total jobs (Natural Breaks)", fontsize=15)
ax.axis('off')

plt.show()

In [ ]:
census[["tp_tyopy"]].sum()

In [ ]:
census_turku[["tp_tyopy"]].sum()

In [ ]:
census_tampere[["tp_tyopy"]].sum()

In [ ]:
census_oulu[["tp_tyopy"]].sum()

In [ ]:

# Reproject area to match census CRS
area_proj = area.to_crs(census.crs)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))

# Census polygons (grid)
census.plot(ax=ax, facecolor="lightblue", edgecolor="black", linewidth=0.5, alpha=0.6, label="Census grid")

# Hexagons
area_proj.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=1, alpha=0.8, label="Hexagons")

ax.set_title("Census grid vs. Hexagons", fontsize=15)
ax.legend()
ax.axis("off")
plt.show()

In [ ]:
census = gpd.read_file("data/grid_data/rttk250m_tilv2019.shp")

In [ ]:
census_turku = gpd.read_file("data/grid_data/rttk250m_tilv2019.shp")

In [ ]:
census_tampere = gpd.read_file("data/grid_data/rttk250m_tilv2019.shp")

In [ ]:
census_oulu = gpd.read_file("data/grid_data/rttk250m_tilv2019.shp")

In [ ]:
area = gpd.read_file('./data/h3_polygons_Helsinki_whole.gpkg')
# Reproject area to match census CRS
area_proj = area.to_crs(census.crs)

In [ ]:
from shapely.geometry import Polygon
area_turku = gpd.read_file('./data/Turku_region_boundary.geojson')

# Function to polyfill a single polygon
def polyfill_geometry(geom, res=9):
    return list(h3.polyfill(geom.__geo_interface__, res, geo_json_conformant=True))

# Prepare set for hex IDs
hex_ids = set()

# Iterate over geometries
for geom in area_turku.geometry:
    # If multipolygon, iterate over polygons
    if geom.geom_type == "MultiPolygon":
        for poly in geom.geoms:
            hex_ids.update(polyfill_geometry(poly, res=9))
    elif geom.geom_type == "Polygon":
        hex_ids.update(polyfill_geometry(geom, res=9))
    else:
        print(f"Skipping geometry type: {geom.geom_type}")

# Convert H3 IDs to polygons
hex_geoms_turku = [Polygon(h3.h3_to_geo_boundary(h, geo_json=True)) for h in hex_ids]

# Create GeoDataFrame with ID
area_turku_hex = gpd.GeoDataFrame(
    {"ID": list(hex_ids)},
    geometry=hex_geoms_turku,
    crs="EPSG:4326"
)

print(area_turku_hex.head())

In [ ]:
area_turku = area_turku_hex.copy()
area_proj_turku = area_turku.to_crs(census.crs)

In [ ]:
from shapely.geometry import Polygon
area_tampere = gpd.read_file('./data/Tampere_region_boundary.geojson')

# Function to polyfill a single polygon
def polyfill_geometry(geom, res=9):
    return list(h3.polyfill(geom.__geo_interface__, res, geo_json_conformant=True))

# Prepare set for hex IDs
hex_ids = set()

# Iterate over geometries
for geom in area_tampere.geometry:
    # If multipolygon, iterate over polygons
    if geom.geom_type == "MultiPolygon":
        for poly in geom.geoms:
            hex_ids.update(polyfill_geometry(poly, res=9))
    elif geom.geom_type == "Polygon":
        hex_ids.update(polyfill_geometry(geom, res=9))
    else:
        print(f"Skipping geometry type: {geom.geom_type}")

# Convert H3 IDs to polygons
hex_geoms_tampere = [Polygon(h3.h3_to_geo_boundary(h, geo_json=True)) for h in hex_ids]

# Create GeoDataFrame with ID
area_tampere_hex = gpd.GeoDataFrame(
    {"ID": list(hex_ids)},
    geometry=hex_geoms_tampere,
    crs="EPSG:4326"
)

print(area_tampere_hex.head())

In [ ]:
area_tampere = area_tampere_hex.copy()
area_proj_tampere = area_tampere.to_crs(census.crs)

In [ ]:
from shapely.geometry import Polygon
area_oulu = gpd.read_file('./data/oulu_region_boundary.geojson')

# Function to polyfill a single polygon
def polyfill_geometry(geom, res=9):
    return list(h3.polyfill(geom.__geo_interface__, res, geo_json_conformant=True))

# Prepare set for hex IDs
hex_ids = set()

# Iterate over geometries
for geom in area_oulu.geometry:
    # If multipolygon, iterate over polygons
    if geom.geom_type == "MultiPolygon":
        for poly in geom.geoms:
            hex_ids.update(polyfill_geometry(poly, res=9))
    elif geom.geom_type == "Polygon":
        hex_ids.update(polyfill_geometry(geom, res=9))
    else:
        print(f"Skipping geometry type: {geom.geom_type}")

# Convert H3 IDs to polygons
hex_geoms_oulu = [Polygon(h3.h3_to_geo_boundary(h, geo_json=True)) for h in hex_ids]

# Create GeoDataFrame with ID
area_oulu_hex = gpd.GeoDataFrame(
    {"ID": list(hex_ids)},
    geometry=hex_geoms_oulu,
    crs="EPSG:4326"
)

print(area_oulu_hex.head())

In [ ]:
area_oulu = area_oulu_hex.copy()
area_proj_oulu = area_oulu.to_crs(census.crs)

In [ ]:
# Intersections
overlay = gpd.overlay(area_proj, census, how="intersection")

# Calculate intersection area
overlay["inter_area"] = overlay.geometry.area

# Weight population by share of area
overlay["weighted_tyo"] = overlay["tp_tyopy"] * (overlay["inter_area"] / overlay.groupby("id_nro")["inter_area"].transform("sum"))

# Aggregate back to hexagons
hex_with_census = overlay.groupby("ID").agg(
    {"weighted_tyo": "sum"}  # keep hex geometry
)

In [ ]:
# Intersections
overlay_turku = gpd.overlay(area_proj_turku, census_clip_turku, how="intersection")

# Calculate intersection area
overlay_turku["inter_area"] = overlay_turku.geometry.area

# Weight jobs by share of area
overlay_turku["weighted_tyo"] = (
    overlay_turku["tp_tyopy"] *
    (overlay_turku["inter_area"] / overlay_turku.groupby("id_nro")["inter_area"].transform("sum"))
)

# Aggregate back to hexagons
hex_with_census_turku = overlay_turku.groupby("ID").agg(
    {"weighted_tyo": "sum"}
).reset_index()

In [ ]:
hex_with_census_turku.sort_values("weighted_tyo")

In [ ]:
# Intersections
overlay_tampere = gpd.overlay(area_proj_tampere, census_clip_tampere, how="intersection")

# Calculate intersection area
overlay_tampere["inter_area"] = overlay_tampere.geometry.area

# Weight jobs by share of area
overlay_tampere["weighted_tyo"] = (
    overlay_tampere["tp_tyopy"] *
    (overlay_tampere["inter_area"] / overlay_tampere.groupby("id_nro")["inter_area"].transform("sum"))
)

# Aggregate back to hexagons
hex_with_census_tampere = overlay_tampere.groupby("ID").agg(
    {"weighted_tyo": "sum"}
).reset_index()

In [ ]:
hex_with_census_tampere.sort_values("weighted_tyo")

In [ ]:
# Intersections
overlay_oulu = gpd.overlay(area_proj_oulu, census_clip_oulu, how="intersection")

# Calculate intersection area
overlay_oulu["inter_area"] = overlay_oulu.geometry.area

# Weight jobs by share of area
overlay_oulu["weighted_tyo"] = (
    overlay_oulu["tp_tyopy"] *
    (overlay_oulu["inter_area"] / overlay_oulu.groupby("id_nro")["inter_area"].transform("sum"))
)

# Aggregate back to hexagons
hex_with_census_oulu = overlay_oulu.groupby("ID").agg(
    {"weighted_tyo": "sum"}
).reset_index()

In [ ]:
hex_with_census_oulu.sort_values("weighted_tyo")

In [ ]:
# Convert H3 ids to hexagon geometries
hex_with_census["geometry"] = hex_with_census.index.to_series().apply(
    lambda h: Polygon(h3.h3_to_geo_boundary(str(h), geo_json=True))
)

# Convertir en GeoDataFrame
area_with_id = gpd.GeoDataFrame(hex_with_census, geometry="geometry", crs="EPSG:4326")

# Visualise with .explore()
area_with_id.explore(column="weighted_tyo", cmap="viridis", legend=True)

In [ ]:

# Convert H3 ids to hexagon geometries
hex_with_census_turku["geometry"] = hex_with_census_turku["ID"].apply(
    lambda h: Polygon(h3.h3_to_geo_boundary(str(h), geo_json=True))
)

# Convertir en GeoDataFrame
area_with_id_turku = gpd.GeoDataFrame(hex_with_census_turku, geometry="geometry", crs="EPSG:4326")

# Visualise with .explore()
area_with_id_turku.explore(column="weighted_tyo", cmap="viridis", legend=True)

In [ ]:
hex_with_census_turku

In [ ]:
# Convert H3 ids to hexagon geometries
hex_with_census_tampere["geometry"] = hex_with_census_tampere["ID"].apply(
    lambda h: Polygon(h3.h3_to_geo_boundary(str(h), geo_json=True))
)

# Convertir en GeoDataFrame
area_with_id_tampere = gpd.GeoDataFrame(hex_with_census_tampere, geometry="geometry", crs="EPSG:4326")

# Visualise with .explore()
area_with_id_tampere.explore(column="weighted_tyo", cmap="viridis", legend=True)

In [ ]:
# Convert H3 ids to hexagon geometries
hex_with_census_oulu["geometry"] = hex_with_census_oulu["ID"].apply(
    lambda h: Polygon(h3.h3_to_geo_boundary(str(h), geo_json=True))
)

# Convertir en GeoDataFrame
area_with_id_oulu = gpd.GeoDataFrame(hex_with_census_oulu, geometry="geometry", crs="EPSG:4326")

# Visualise with .explore()
area_with_id_oulu.explore(column="weighted_tyo", cmap="viridis", legend=True)

In [ ]:
area_with_id.to_parquet("data/job_distribution_from_census.parquet")

In [ ]:
area_with_id_turku.to_parquet("data/job_distribution_from_census_turku.parquet")

In [ ]:
area_with_id_tampere.to_parquet("data/job_distribution_from_census_turku.parquet")

In [ ]:
area_with_id_oulu.to_parquet("data/job_distribution_from_census_oulu.parquet")

In [ ]:
#hex_with_census.set_crs(crs=area_proj.crs, allow_override=True)